# ML-09 — Temporal validation and research claim audit

This capstone audits its own two measured findings. No external FlyRank research paper was supplied, so no finding is attributed to an unseen paper. The June evaluation, raw-data quality amendment and code were recorded in the repository. The original method was committed before the first June read.

## 1. Two findings and the questions they raise

**Finding 1:** On the final June cohort, pooled precision@20 is 85% for the tree versus 75% for the volume baseline. Are these twenty pages independent, and is the model simply concentrating on particular clients?

**Finding 2:** The model performs worse within all five clients with more than twenty eligible pages. Does a global ranking metric support the operational decision if review time is allocated by client?

The proxy comes from later impressions, not an editor's decision or a refresh experiment. Both findings concern visibility decline only. The sixth client has seventeen pages, so selecting all seventeen cannot test ranking quality.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "skills/README.md").exists())
sys.path.insert(0, str(ROOT / "work/scripts"))
import capstone_pipeline as cp
SUMMARY_PATH = ROOT / "work/outputs/capstone_metrics.json"
if not SUMMARY_PATH.exists():
    cp.execute()
s = json.loads(SUMMARY_PATH.read_text())
print("Dataset revision:", s["dataset_revision"])
print("Protocol:", s["protocol_version"], s["protocol_sha256"])

display(pd.DataFrame(s["june_primary"]).T)
display(pd.DataFrame(s["per_client_primary"]))

/var/home/tanzimul/Repos/github.com/tanzimul3islam/flyrank-ml-internship-starter/work/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Protocol: capstone_v1 181f2c24f2742facb0f37fdf77f1ab5727a2b5cfa5876477fe235e63117e437f


,n,clients,base_rate,k,top_k_declines,precision_at_k,roc_auc,average_precision,top_k_clients
baseline,35588.0,6.0,0.588963,20.0,15.0,0.75,0.598933,0.635356,4.0
model,35588.0,6.0,0.588963,20.0,17.0,0.85,0.575267,0.639196,2.0


,anonymous_client,n,k,base_rate,baseline_p_at_k,model_p_at_k
0,Group 1,109,20,0.688073,0.900000,0.750000
1,Group 2,9695,20,0.683858,0.950000,0.850000
2,Group 3,8930,20,0.541769,0.650000,0.200000
3,Group 4,14579,20,0.546883,0.600000,0.250000
4,Group 5,17,17,0.470588,0.470588,0.470588
5,Group 6,2258,20,0.635961,0.550000,0.400000


## 2. Before/after: reused March comparison to June final evaluation

The model is fixed. March's comparison clients were held out of training but their results had been seen in previous weeks. June's primary cohort uses only those untrained clients at a later decision date. Six of the original eight have eligible labeled pages. No June label informed fitting, feature selection or parameter tuning.

A data-only amendment removed 6,390 exact duplicate June page-day copies after the grain check failed and before model scoring. No conflicting records were collapsed. This change is explicitly documented, rather than hidden as a perfect preregistered data pipeline.

The 500-resample paired client bootstrap reranks the global top twenty within each resampled client pool. Its -60 to +20 percentage-point interval is descriptive and wide; six clusters do not justify a precise generalization claim.

In [2]:
rows=[]
for cohort in ["march_comparison","june_primary","june_secondary_all_clients"]:
    for name,metrics in s[cohort].items():rows.append({"cohort":cohort,"method":name,**metrics})
display(pd.DataFrame(rows))
display(pd.DataFrame([s["bootstrap"]]))
display(pd.DataFrame([s["march_source"],s["june_source"]]))
print("Data-only amendment:")
display(json.loads((ROOT/"work/capstone_data_amendment.json").read_text()))

,cohort,method,n,clients,base_rate,k,top_k_declines,precision_at_k,roc_auc,average_precision,top_k_clients
0,march_comparison,baseline,43487,8,0.374365,20,9,0.45,0.532410,0.398072,3
1,march_comparison,model,43487,8,0.374365,20,14,0.70,0.603773,0.445687,2
2,june_primary,baseline,35588,6,0.588963,20,15,0.75,0.598933,0.635356,4
3,june_primary,model,35588,6,0.588963,20,17,0.85,0.575267,0.639196,2
4,june_secondary_all_clients,baseline,64818,42,0.502900,20,17,0.85,0.584050,0.547044,7
5,june_secondary_all_clients,model,64818,42,0.502900,20,16,0.80,0.578351,0.558043,2


,method,resamples,clusters,lift_interval_95,caution
0,paired client bootstrap; rerank global top20; ...,500,6,"[-0.6000000000000001, 0.19999999999999996]",Descriptive percentile interval with few clien...


,source_rows,first_date,last_date,clients,gsc_available_rows,gsc_false_rows,gsc_null_rows,subone_position_rows,duplicate_keys,identical_extra_rows_removed,conflicting_keys,clean_page_day_rows,partition,source_grain_violations_before_cleaning,source_grain_violations_after_cleaning,aggregate_rows
0,9841378,2026-03-01,2026-03-31,55,3611061,6230317,0,101548,0,0,0,9841378,fact_content_daily_performance/month=2026-03/d...,0,0,176738
1,11694072,2026-06-01,2026-06-30,65,3878937,7815135,0,38010,6390,6390,0,11687682,fact_content_daily_performance/month=2026-06/d...,6390,0,208636


Data-only amendment:


{'date': '2026-09-26',
 'original_protocol_commit': '01a5c7b',
 'stage': 'Before any June model metric was calculated',
 'reason': 'June source grain check found 6390 duplicated page-day keys, each with multiplicity 2, no null keys, and zero conflicts across all fields used by this analysis.',
 'change': 'Collapse only identical copies across the selected daily fields; conflicting duplicates continue to fail. Apply the same quality rule to both months.',
 'unchanged': 'Model, feature definitions, eligibility, label, split, thresholds and comparison metrics remain frozen.',
 'June_extra_rows_removed': 6390,
 'June_conflicting_keys': 0}

## 3. Leakage audit and feature stability

The executable check below mutates all future-window counts and coverage in a copy of June's aggregated data. Past eligibility and all five feature values must remain unchanged. The target and label availability may change, but those outputs are never model inputs.

Additional safeguards: disjoint March training/final clients, train-only imputation, no hash IDs or target-derived columns in the feature list, no fixed-window query table, and June held out until the model was fixed. These checks establish code boundaries; they cannot establish when upstream source values first became available.

In [3]:
raw, _ = cp.load_month("2026-06")
original, X = cp.prepare(raw,"2026-06")
perturbed=raw.copy()
perturbed["future_impressions"]=0
perturbed["future_valid_days"]=0
changed, X_changed=cp.prepare(perturbed,"2026-06")
pd.testing.assert_frame_equal(X,X_changed)
pd.testing.assert_frame_equal(original[cp.KEYS],changed[cp.KEYS])
assert changed.declined.isna().all()
assert all(s["checks"].values())
assert not any("future" in f or "declin" in f or "hash" in f for f in cp.FEATURES)
print("PASS: changing outcomes does not change score inputs or past eligibility.")
display(pd.DataFrame(s["feature_summary"]))
display(pd.DataFrame(s["permutation_interpretation"]))

PASS: changing outcomes does not change score inputs or past eligibility.


,feature,train_median,final_median,train_missing_fraction,final_missing_fraction
0,mean_daily_impressions,35.928571,38.857143,0.000000,0.0
1,ctr_pct,0.196592,0.255346,0.000000,0.0
2,impression_weighted_position,8.797459,9.222756,0.000354,0.0
3,active_day_share,1.000000,1.000000,0.000000,0.0
4,daily_impression_cv,0.425062,0.391861,0.000000,0.0


,feature,mean_auc_drop,std_auc_drop
0,mean_daily_impressions,0.062836,0.001524
1,ctr_pct,0.025633,0.002267
2,impression_weighted_position,-0.000369,0.000608
3,active_day_share,0.000000,0.000000
4,daily_impression_cv,0.005300,0.001317


## 4. Rewrite the claim

**Too strong:** “The model improves content recommendations and will recover lost traffic.”

**Supported:** “In a frozen retrospective June test on six untrained clients, the model found two more declining pages in a pooled top twenty than the volume baseline. Its within-client rankings were worse in every rankable client, its AUC was lower, and the client-bootstrap interval was wide. The result does not establish editorial usefulness or causal refresh benefit.”

We would not broadly deploy the tree based on this evaluation. Collect independent editorial judgments and prospectively evaluate the actual review-allocation policy first. Preserve the frozen results as a negative/qualified finding rather than tuning the June test into a success.

In [4]:
groups=pd.DataFrame(s["per_client_primary"])
rankable=groups.loc[groups.n.gt(groups.k)]
assert (rankable.model_p_at_k < rankable.baseline_p_at_k).all()
print(f"Model loses within all {len(rankable)} rankable clients.")
print("Unknown final outcomes:",s["june_primary_unlabeled_pages"])
print("No claim of causal refresh impact or confirmed editorial value.")

Model loses within all 5 rankable clients.
Unknown final outcomes: 3808
No claim of causal refresh impact or confirmed editorial value.


## 5. Self-check

- [x] Distinguishes reused March evidence from frozen June evaluation.
- [x] Reports identical-cohort baselines, prevalence, adverse client results and uncertainty.
- [x] Documents the data-quality amendment before outcome scoring.
- [x] Executes an outcome-mutation leakage check.
- [x] Rewrites claims around the observed limits; no external paper findings invented.
- [x] Executed outputs saved and backed by the committed aggregate receipt.